# Similaridade entre topicos da pauta e do discurso
Este notebook gera visualizacoes comparando a similaridade entre topicos da pauta e topicos do discurso antes e depois da eleicao, por partido, para os fluxos v1 e v2.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BASE_DIR = Path.cwd().parents[1]
DATA_DIR = BASE_DIR / "data" / "party_agenda" / "embeddings"
PARTIES = ["MDB", "NOVO", "PL", "PSOL", "PT", "UNIAO"]
FLOWS = ["v1", "v2"]
ELECTION_PERIODS = ["antesDaEleicao", "depoisDaEleicao"]
CSV_NAME = "similaridade_topics_discurso_topics_agenda.csv"

def load_similarity_table(party: str, flow: str, period: str) -> pd.DataFrame:
    file_path = DATA_DIR / party / "similaridade" / "topics" / "lda" / flow / period / CSV_NAME
    if not file_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(file_path)
    df["party"] = party
    df["flow"] = flow
    df["period"] = period
    return df

rows = []
for party in PARTIES:
    for flow in FLOWS:
        for period in ELECTION_PERIODS:
            rows.append(load_similarity_table(party, flow, period))

similarity_df = pd.concat([df for df in rows if not df.empty], ignore_index=True)
similarity_df.head()

In [ ]:
if similarity_df.empty:
    raise ValueError("Nenhum arquivo foi encontrado. Verifique os caminhos dos CSVs.")

similarity_df["cosine_similarity"] = pd.to_numeric(similarity_df["cosine_similarity"], errors="coerce")
summary = (
    similarity_df
    .groupby(["party", "flow", "period"], as_index=False)
    .agg(mean_similarity=("cosine_similarity", "mean"),
         median_similarity=("cosine_similarity", "median"),
         max_similarity=("cosine_similarity", "max"),
         rows=("cosine_similarity", "size"))
)
summary

In [ ]:
fig_v1 = px.bar(
    summary[summary["flow"] == "v1"],
    x="party",
    y="mean_similarity",
    color="period",
    barmode="group",
    title="Similaridade media entre topicos de pauta e discurso (v1)",
    labels={"mean_similarity": "Similaridade media", "period": "Periodo"},
    text=summary.loc[summary["flow"] == "v1", "mean_similarity"].round(3).astype(str),
)
fig_v1.update_layout(height=420, legend_title_text="Periodo")
fig_v1.update_traces(textposition="outside")
fig_v1.show()

In [ ]:
fig_v2 = px.bar(
    summary[summary["flow"] == "v2"],
    x="party",
    y="mean_similarity",
    color="period",
    barmode="group",
    title="Similaridade media entre topicos de pauta e discurso (v2)",
    labels={"mean_similarity": "Similaridade media", "period": "Periodo"},
    text=summary.loc[summary["flow"] == "v2", "mean_similarity"].round(3).astype(str),
)
fig_v2.update_layout(height=420, legend_title_text="Periodo")
fig_v2.update_traces(textposition="outside")
fig_v2.show()

In [ ]:
# Tabela comparativa dos topicos mais similares
top_matches = (
    similarity_df
    .sort_values(["party", "flow", "period", "cosine_similarity"], ascending=[True, True, True, False])
    .groupby(["party", "flow", "period"], as_index=False)
    .head(5)
    .reset_index(drop=True)
)
top_matches.head()

In [ ]:
fig = px.scatter(
    top_matches,
    x="agenda_topic",
    y="discourse_topic",
    size="cosine_similarity",
    color="period",
    facet_row="party",
    facet_col="flow",
    title="Topicos mais similares entre pauta e discurso (top 5 por partido/fluxo/periodo)",
    labels={"agenda_topic": "Topico de pauta", "discourse_topic": "Topico de discurso"},
    hover_data=["cosine_similarity", "agenda_terms", "discourse_terms"],
)
fig.update_layout(height=220 * len(PARTIES))
fig.show()

In [ ]:
# Topicos do UNIAO — v1 (antes e depois)
uniao_v1 = (
    similarity_df[(similarity_df["party"] == "UNIAO") & (similarity_df["flow"] == "v1")]
    .sort_values(["period", "cosine_similarity"], ascending=[True, False])
    [["period", "agenda_topic", "agenda_terms", "discourse_topic", "discourse_terms", "cosine_similarity"]]
)
if uniao_v1.empty:
    raise ValueError("Sem dados para UNIAO no fluxo v1.")

import re as _re

def _format_terms(terms_str: str) -> str:
    """Extract term labels from weighted string; return bullet list."""
    matches = _re.findall(r'\d+\.\d+\*"([^"]*)"', str(terms_str))
    terms = [t.strip() for t in matches if t.strip()]
    if terms:
        return "\n".join(f"\u2022 {t.strip()}" for t in terms[:10])
    return str(terms_str)

_disp = uniao_v1.copy()
_disp["cosine_similarity"] = _disp["cosine_similarity"].round(3)
_disp["agenda_terms"] = _disp["agenda_terms"].apply(_format_terms)
_disp["discourse_terms"] = _disp["discourse_terms"].apply(_format_terms)
_disp.columns = ['Período', 'Tóp. Pauta', 'Termos Pauta', 'Tóp. Discurso', 'Termos Discurso', 'Similaridade']

_n = len(_disp)
_n_cols = len(_disp.columns)
_row_colors = ["#f2f4f8" if i % 2 == 0 else "white" for i in range(_n)]

fig_uniao_v1 = go.Figure(
    data=[
        go.Table(
            columnwidth=[100, 60, 310, 60, 310, 80],
            header=dict(
                values=list(_disp.columns),
                fill_color="#2c3e50",
                font=dict(color="white", size=12, family="Arial"),
                align=["left", "center", "left", "center", "left", "center"],
                height=38,
                line=dict(color="#2c3e50", width=1),
            ),
            cells=dict(
                values=[_disp[col] for col in _disp.columns],
                fill_color=[_row_colors] * _n_cols,
                align=["left", "center", "left", "center", "left", "center"],
                font=dict(size=11, family="Arial"),
                height=180,
                line=dict(color="#ced4da", width=1),
            ),
        )
    ]
)
fig_uniao_v1.update_layout(
    title=dict(text="Tópicos de pauta vs. discurso — UNIAO, fluxo v1", font=dict(size=14, family="Arial")),
    height=max(560, 38 + _n * 180 + 60),
    margin=dict(l=8, r=8, t=60, b=8),
)
fig_uniao_v1.show()

In [ ]:
# Topicos do UNIAO — v2 (antes e depois)
uniao_v2 = (
    similarity_df[(similarity_df["party"] == "UNIAO") & (similarity_df["flow"] == "v2")]
    .sort_values(["period", "cosine_similarity"], ascending=[True, False])
    [["period", "agenda_topic", "agenda_terms", "discourse_topic", "discourse_terms", "cosine_similarity"]]
)
if uniao_v2.empty:
    raise ValueError("Sem dados para UNIAO no fluxo v2.")

import re as _re

def _format_terms(terms_str: str) -> str:
    """Extract term labels from weighted string; return bullet list."""
    matches = _re.findall(r'\d+\.\d+\*"([^"]*)"', str(terms_str))
    terms = [t.strip() for t in matches if t.strip()]
    if terms:
        return "\n".join(f"\u2022 {t.strip()}" for t in terms[:10])
    return str(terms_str)

_disp = uniao_v2.copy()
_disp["cosine_similarity"] = _disp["cosine_similarity"].round(3)
_disp["agenda_terms"] = _disp["agenda_terms"].apply(_format_terms)
_disp["discourse_terms"] = _disp["discourse_terms"].apply(_format_terms)
_disp.columns = ['Período', 'Tóp. Pauta', 'Termos Pauta', 'Tóp. Discurso', 'Termos Discurso', 'Similaridade']

_n = len(_disp)
_n_cols = len(_disp.columns)
_row_colors = ["#f2f4f8" if i % 2 == 0 else "white" for i in range(_n)]

fig_uniao_v2 = go.Figure(
    data=[
        go.Table(
            columnwidth=[100, 60, 310, 60, 310, 80],
            header=dict(
                values=list(_disp.columns),
                fill_color="#2c3e50",
                font=dict(color="white", size=12, family="Arial"),
                align=["left", "center", "left", "center", "left", "center"],
                height=38,
                line=dict(color="#2c3e50", width=1),
            ),
            cells=dict(
                values=[_disp[col] for col in _disp.columns],
                fill_color=[_row_colors] * _n_cols,
                align=["left", "center", "left", "center", "left", "center"],
                font=dict(size=11, family="Arial"),
                height=180,
                line=dict(color="#ced4da", width=1),
            ),
        )
    ]
)
fig_uniao_v2.update_layout(
    title=dict(text="Tópicos de pauta vs. discurso — UNIAO, fluxo v2", font=dict(size=14, family="Arial")),
    height=max(560, 38 + _n * 180 + 60),
    margin=dict(l=8, r=8, t=60, b=8),
)
fig_uniao_v2.show()

In [ ]:
# Tabela com topicos mais similares para todos os partidos (v1 e v2)

def render_topic_table(df: pd.DataFrame, title: str) -> None:
    if df.empty:
        raise ValueError(f"Sem dados para {title}.")
    df = df.copy()
    df["cosine_similarity"] = df["cosine_similarity"].round(3)
    fig = go.Figure(
        data=[
            go.Table(
                header=dict(values=list(df.columns), fill_color="#e9ecef", align="left"),
                cells=dict(values=[df[col] for col in df.columns], align="left"),
            )
        ]
    )
    fig.update_layout(title=title, height=420)
    fig.show()

all_v1 = (
    top_matches[top_matches["flow"] == "v1"]
    .sort_values(["party", "period", "cosine_similarity"], ascending=[True, True, False])
    [["party", "period", "agenda_topic", "discourse_topic", "cosine_similarity"]]
)
all_v2 = (
    top_matches[top_matches["flow"] == "v2"]
    .sort_values(["party", "period", "cosine_similarity"], ascending=[True, True, False])
    [["party", "period", "agenda_topic", "discourse_topic", "cosine_similarity"]]
)

render_topic_table(all_v1, "Topicos mais similares por partido (v1)")
render_topic_table(all_v2, "Topicos mais similares por partido (v2)")

In [ ]:
# Tabela resumida para exportacao ou inspecao
summary_table = summary.sort_values(["party", "flow", "period"])
summary_table